In [54]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

In [55]:
load_dotenv()  # Load environment variables from .env file

True

In [56]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [57]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    eval_score: float
    

In [58]:
def create_outline(state: BlogState) -> BlogState:
    #fetch the title from the state
    title = state['title']
    #create a prompt to generate an outline
    prompt = f"Create a detailed outline for a blog post titled: '{title}'"
    #use the model to generate the outline
    outline = model.invoke(prompt)
    #update the state with the generated outline
    state['outline'] = outline
    return state

In [59]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    #fetch the outline from the state
    outline = state['outline']
    #create a prompt to generate the blog content
    prompt = f"Write a comprehensive blog post based on the title '{title}' using following outline:\n{outline}"
    #use the model to generate the blog content
    content = model.invoke(prompt)
    #update the state with the generated content
    state['content'] = content
    return state

In [60]:
def evaluate_blog(state: BlogState) -> BlogState:
    content = state['content']
    outline = state['outline']
    #create a prompt to evaluate the blog content
    prompt = f"Evaluate the following blog post content for clarity, engagement, and informativeness:\n{content}\n use the blog outline {outline} as a reference. Provide a score from 1 to 10. only give me the score."
    #use the model to evaluate the content
    eval_score = model.invoke(prompt).content.strip()
    #update the state with the evaluation score
    state['eval_score'] = float(eval_score)
    return state

In [61]:
graph = StateGraph(BlogState)
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

In [62]:
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

In [63]:
workflow = graph.compile()

In [64]:
initial_state = {'title': 'The Future of AI in Everyday Life'}

In [65]:
workflow.invoke(initial_state)
final_state = workflow.invoke(initial_state) 

In [66]:
print(final_state)

{'title': 'The Future of AI in Everyday Life', 'outline': AIMessage(content='**Title:** The Future of AI in Everyday Life\n\n**I. Introduction**\n\n- Brief overview of the rapid advancements in AI technology\n- Importance of understanding the role of AI in our daily lives\n- Thesis statement: As AI continues to evolve, it will become increasingly intertwined with our daily routines, transforming the way we live, work, and interact with one another.\n\n**II. Predictions for AI in the Home**\n\n- Smart homes and voice assistants (e.g., Alexa, Google Home)\n- AI-powered home security systems and surveillance\n- Personalized recommendations and automated household tasks\n- Smart appliances and cooking assistants\n- Examples of current and future AI-powered home devices\n\n**III. AI in Healthcare**\n\n- Telemedicine and virtual consultations\n- AI-assisted diagnosis and treatment planning\n- Personalized medicine and tailored treatment options\n- AI-powered healthcare assistants and chatbot

In [68]:
print(final_state['content'])
print(final_state['eval_score'])

content='**The Future of AI in Everyday Life**\n\nAs we navigate the complexities of the modern world, it\'s becoming increasingly clear that Artificial Intelligence (AI) will play a pivotal role in shaping our daily lives. From the humble beginnings of AI-powered virtual assistants to the sophisticated applications of AI in industries such as healthcare and transportation, the technology has come a long way in a relatively short period.\n\nIn this blog post, we\'ll take a comprehensive look at the future of AI in everyday life, exploring its potential applications, benefits, and challenges. We\'ll examine how AI is transforming the way we live, work, and interact with one another, and what this means for individuals, businesses, and governments.\n\n**II. Predictions for AI in the Home**\n\nThe home is one of the most intimate and personal spaces in our lives, and AI is set to revolutionize the way we interact with it. Some of the most exciting predictions for AI in the home include:\n